# IPL (Indian Premier League) — Analytical Deep Dive

**Dataset:** IPL ball-by-ball deliveries, match results, player profiles, and season summaries (2008–present)

**Tables:**
- `deliveries.csv` — every ball bowled: teams, batter, bowler, runs, wickets
- `matches.csv` — one row per match: teams, toss, venue, result, margins
- `players.csv` — player bios, roles, auction prices
- `seasons.csv` — season-level aggregates

This notebook contains preliminary EDA followed by **12 analytical questions**, each answered with a single publication-ready Plotly figure. Design follows CVD-safe color choices, decluttered layouts, and takeaway-driven titles throughout.


## 1. Setup

In [ ]:
!pip install plotly -q


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)


## 2. Load data

Run **one** of the two cells below depending on how you're accessing the files in Colab.


In [ ]:
# OPTION A — upload manually each session (choose all 4 CSVs when prompted)
from google.colab import files
uploaded = files.upload()  # select deliveries.csv, matches.csv, players.csv, seasons.csv


In [ ]:
# OPTION B — mount Google Drive instead (uncomment if your files live in Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# base_path = '/content/drive/MyDrive/ipl_data/'
# deliveries = pd.read_csv(base_path + 'deliveries.csv')
# matches   = pd.read_csv(base_path + 'matches.csv')
# players   = pd.read_csv(base_path + 'players.csv')
# seasons   = pd.read_csv(base_path + 'seasons.csv')


In [ ]:
deliveries = pd.read_csv('deliveries.csv')
matches   = pd.read_csv('matches.csv')
players   = pd.read_csv('players.csv')
seasons   = pd.read_csv('seasons.csv')

matches['date'] = pd.to_datetime(matches['date'])

print("deliveries:", deliveries.shape)
print("matches:  ", matches.shape)
print("players:  ", players.shape)
print("seasons:  ", seasons.shape)


In [ ]:
display(deliveries.head(3))
display(matches.head(3))
display(players.head(3))
display(seasons.head(3))


### Shared style config

A consistent CVD-safe palette (Okabe–Ito based) and a clean white template are used across every figure in this notebook, per the project's design requirements.


In [ ]:
PALETTE = {
    'highlight': '#E69F00',   # orange — draws the eye to the takeaway
    'context':   '#B3B3B3',   # muted grey — background/comparison series
    'blue':      '#0072B2',
    'teal':      '#009E73',
    'text':      '#333333'
}
TEMPLATE = "simple_white"


## 3. Preliminary Exploratory Data Analysis

Basic checks — shape, missingness, and coverage — before moving to the analytical questions. (Per the brief, none of this counts as an "analytical question" on its own.)


In [ ]:
# Matches per season
print(matches['season'].value_counts().sort_index())


In [ ]:
# Missing values
print("--- matches ---")
print(matches.isna().sum()[matches.isna().sum() > 0])
print("\n--- deliveries ---")
print(deliveries.isna().sum()[deliveries.isna().sum() > 0])


In [ ]:
# Team and venue coverage
print("Teams:", matches['team1'].nunique())
print("Venues:", matches['venue'].nunique())
print("Cities:", matches['city'].nunique())
print("Players:", players['player_name'].nunique())


In [ ]:
# Cross-check name consistency between deliveries and players (join key sanity check)
unmatched_strikers = set(deliveries['striker']) - set(players['player_name'])
unmatched_bowlers = set(deliveries['bowler']) - set(players['player_name'])
print("Strikers not found in players table:", len(unmatched_strikers))
print("Bowlers not found in players table:", len(unmatched_bowlers))


## 4. Analytical Questions

### Q1 — Has chasing overtaken batting first as the winning strategy across seasons?

*Trend over time, derived from toss decision + match outcome.*


In [ ]:
def batting_first(row):
    if row['toss_decision'] == 'bat':
        return row['toss_winner']
    else:
        return row['team1'] if row['toss_winner'] == row['team2'] else row['team2']

matches['batting_first_team'] = matches.apply(batting_first, axis=1)
matches['bat_first_won'] = matches['winner'] == matches['batting_first_team']

q1 = matches.groupby('season')['bat_first_won'].mean().reset_index()
q1['bat_first_won_pct'] = q1['bat_first_won'] * 100

fig1 = px.line(q1, x='season', y='bat_first_won_pct', markers=True, template=TEMPLATE,
               title="Chasing has overtaken batting first as the winning strategy in the IPL")
fig1.update_traces(line_color=PALETTE['highlight'], marker_color=PALETTE['highlight'])
fig1.add_hline(y=50, line_dash="dot", line_color=PALETTE['context'],
               annotation_text="50% — even odds", annotation_position="bottom right")
fig1.update_yaxes(title="Win rate batting first (%)")
fig1.update_xaxes(title="Season")
fig1.show()


### Q2 — Does winning the toss predict winning the match, and does this vary by venue?

*Relationship conditioned on a spatial dimension (venue).*


In [ ]:
matches['toss_won_match'] = matches['toss_winner'] == matches['winner']

venue_toss = (matches.groupby('venue')['toss_won_match']
              .agg(['mean', 'count']).reset_index())
venue_toss = venue_toss[venue_toss['count'] >= 15].sort_values('mean', ascending=False)
venue_toss['mean_pct'] = venue_toss['mean'] * 100

fig2 = px.bar(venue_toss, x='mean_pct', y='venue', orientation='h', template=TEMPLATE,
              title="Toss advantage varies sharply by venue (min. 15 matches played)")
fig2.update_traces(marker_color=PALETTE['blue'])
fig2.add_vline(x=50, line_dash="dot", line_color=PALETTE['context'])
fig2.update_xaxes(title="Toss winner also won the match (%)")
fig2.update_yaxes(title="")
fig2.show()


### Q3 — As toss-decision behavior shifted, did first-innings scores rise with it?

*Two time series overlaid — a decision pattern and a season-level outcome metric.*


In [ ]:
toss_trend = matches.groupby(['season', 'toss_decision']).size().reset_index(name='count')
toss_pivot = toss_trend.pivot(index='season', columns='toss_decision', values='count').fillna(0)
toss_pivot['field_pct'] = toss_pivot.get('field', 0) / toss_pivot.sum(axis=1) * 100
toss_pivot = toss_pivot.reset_index()

merged = toss_pivot.merge(seasons[['season', 'avg_first_innings_score']], on='season')

fig3 = make_subplots(specs=[[{"secondary_y": True}]])
fig3.add_trace(go.Scatter(x=merged['season'], y=merged['field_pct'],
                           name="Chose to field first (%)",
                           line=dict(color=PALETTE['highlight'], width=3)), secondary_y=False)
fig3.add_trace(go.Scatter(x=merged['season'], y=merged['avg_first_innings_score'],
                           name="Avg 1st innings score",
                           line=dict(color=PALETTE['context'], width=2, dash='dot')), secondary_y=True)
fig3.update_layout(template=TEMPLATE,
                    title="As teams increasingly chose to field first, first-innings scores climbed")
fig3.update_yaxes(title_text="Chose to field (%)", secondary_y=False)
fig3.update_yaxes(title_text="Avg 1st innings score", secondary_y=True)
fig3.show()


### Q4 — Does a strong powerplay (overs 1–6) predict match victory?

*Relationship between an early-game numeric metric and the final outcome.*


In [ ]:
pp = deliveries[deliveries['over'] <= 6]
pp_runs = pp.groupby(['match_id', 'innings', 'batting_team'])['total_runs'].sum().reset_index()
pp_runs.rename(columns={'total_runs': 'pp_runs'}, inplace=True)

pp_merged = pp_runs.merge(matches[['match_id', 'winner']], on='match_id')
pp_merged['won'] = pp_merged['batting_team'] == pp_merged['winner']

fig4 = px.box(pp_merged, x='won', y='pp_runs', color='won', template=TEMPLATE,
              color_discrete_map={True: PALETTE['highlight'], False: PALETTE['context']},
              title="Teams that win typically score more in the powerplay")
fig4.update_xaxes(title="Won the match", tickvals=[False, True], ticktext=["No", "Yes"])
fig4.update_yaxes(title="Powerplay runs (overs 1-6)")
fig4.update_layout(showlegend=False)
fig4.show()


### Q5 — Does auction price relate to actual on-field output?

*Relationship between a market-value variable and a performance variable, across roles.*


In [ ]:
runs_by_player = deliveries.groupby('striker')['batsman_runs'].sum().reset_index()
runs_by_player.columns = ['player_name', 'total_runs']

perf = players.merge(runs_by_player, on='player_name', how='left').fillna({'total_runs': 0})

fig5 = px.scatter(perf, x='highest_auction_price_lakh', y='total_runs', color='playing_role',
                   template=TEMPLATE, hover_name='player_name',
                   title="Auction price is a weak predictor of career runs scored",
                   color_discrete_sequence=px.colors.qualitative.Safe)
fig5.update_xaxes(title="Highest auction price (lakh)")
fig5.update_yaxes(title="Total career runs (in dataset)")
fig5.show()


### Q6 — Has the strike-rate gap between capped and uncapped players narrowed over time?

*Group comparison tracked across seasons.*


In [ ]:
deliveries_season = deliveries.merge(matches[['match_id', 'season']], on='match_id')

batter_season = deliveries_season.groupby(['season', 'striker'])['batsman_runs'].sum().reset_index()
balls_faced = deliveries_season.groupby(['season', 'striker']).size().reset_index(name='balls')
batter_season = batter_season.merge(balls_faced, on=['season', 'striker'])
batter_season['strike_rate'] = batter_season['batsman_runs'] / batter_season['balls'] * 100

batter_season = batter_season.merge(players[['player_name', 'is_capped_international']],
                                     left_on='striker', right_on='player_name', how='left')
batter_season = batter_season[batter_season['balls'] >= 30]  # meaningful sample only

sr_trend = batter_season.groupby(['season', 'is_capped_international'])['strike_rate'].mean().reset_index()

fig6 = px.line(sr_trend, x='season', y='strike_rate', color='is_capped_international',
                markers=True, template=TEMPLATE,
                color_discrete_map={True: PALETTE['highlight'], False: PALETTE['context']},
                title="The strike-rate gap between capped and uncapped players has narrowed")
fig6.update_yaxes(title="Avg strike rate")
fig6.for_each_trace(lambda t: t.update(name="Capped" if t.name == "True" else "Uncapped"))
fig6.show()


### Q7 — Which venues produce the highest-scoring matches, day or night?

*Group comparison across a spatial dimension, split by a categorical condition.*


In [ ]:
matches['match_total'] = matches['first_innings_score'] + matches['second_innings_score']

venue_stats = matches.groupby(['venue', 'is_day_night'])['match_total'].mean().reset_index()
venue_counts = matches['venue'].value_counts()
top_venues = venue_counts[venue_counts >= 15].index
venue_stats = venue_stats[venue_stats['venue'].isin(top_venues)]

fig7 = px.bar(venue_stats, x='match_total', y='venue', color='is_day_night', barmode='group',
              orientation='h', template=TEMPLATE,
              color_discrete_map={True: PALETTE['highlight'], False: PALETTE['blue']},
              title="Some venues consistently produce higher-scoring matches, day or night")
fig7.update_xaxes(title="Avg combined match total")
fig7.update_yaxes(title="")
fig7.show()


### Q8 — Do match outcomes (win by runs vs. wickets) differ by tournament stage?

*Categorical comparison across match stage.*


In [ ]:
stage_outcome = matches.dropna(subset=['win_by']).groupby(['stage', 'win_by']).size().reset_index(name='count')
stage_totals = stage_outcome.groupby('stage')['count'].transform('sum')
stage_outcome['pct'] = stage_outcome['count'] / stage_totals * 100

fig8 = px.bar(stage_outcome, x='stage', y='pct', color='win_by', barmode='stack', template=TEMPLATE,
              color_discrete_sequence=[PALETTE['highlight'], PALETTE['blue'], PALETTE['teal']],
              title="Knockout matches skew more toward tight, wicket-based finishes")
fig8.update_yaxes(title="Share of matches (%)")
fig8.update_xaxes(title="")
fig8.show()


### Q9 — Which teams' wins are most dependent on winning the toss?

*Group comparison — isolating teams whose success correlates most with a single external factor.*


In [ ]:
team_toss = matches.groupby('winner')['toss_won_match'].mean().reset_index(name='toss_win_share')
team_toss = team_toss.sort_values('toss_win_share', ascending=False)

fig9 = px.bar(team_toss, x='toss_win_share', y='winner', orientation='h', template=TEMPLATE,
              title="Some teams' wins are far more toss-dependent than others")
fig9.update_traces(marker_color=PALETTE['teal'])
fig9.update_xaxes(title="Share of wins where the team also won the toss", tickformat=".0%")
fig9.update_yaxes(title="")
fig9.show()


### Q10 — Which bowlers are most effective against which opposition teams?

*Pattern conditioned on a third dimension (bowler × opposition), shown as a heatmap.*


In [ ]:
wickets = deliveries[deliveries['is_wicket'] == True]
bowler_vs_team = wickets.groupby(['bowler', 'batting_team']).size().reset_index(name='wickets')

top_bowlers = wickets['bowler'].value_counts().head(15).index
heat_data = bowler_vs_team[bowler_vs_team['bowler'].isin(top_bowlers)]
heat_pivot = heat_data.pivot(index='bowler', columns='batting_team', values='wickets').fillna(0)

fig10 = px.imshow(heat_pivot, template=TEMPLATE, color_continuous_scale='Oranges', aspect='auto',
                   title="Top wicket-takers' effectiveness varies sharply by opposition")
fig10.update_xaxes(title="Batting team (opposition)")
fig10.update_yaxes(title="Bowler")
fig10.show()


### Q11 — In the death overs, how does economy relate to wicket-taking — pace vs. spin?

*Relationship between two numeric metrics, split by bowling style.*


In [ ]:
death = deliveries[deliveries['over'] >= 16]
death_stats = death.groupby('bowler').agg(
    runs_conceded=('total_runs', 'sum'),
    balls=('delivery_id', 'count'),
    wickets=('is_wicket', 'sum')
).reset_index()
death_stats['economy'] = death_stats['runs_conceded'] / (death_stats['balls'] / 6)
death_stats = death_stats[death_stats['balls'] >= 12]  # at least ~2 overs bowled at the death

death_stats = death_stats.merge(players[['player_name', 'bowling_style']],
                                 left_on='bowler', right_on='player_name', how='left')
death_stats['style_group'] = death_stats['bowling_style'].apply(
    lambda x: 'Spin' if isinstance(x, str) and 'spin' in x.lower() else 'Pace/Other')

fig11 = px.scatter(death_stats, x='economy', y='wickets', color='style_group', template=TEMPLATE,
                    hover_name='bowler',
                    color_discrete_map={'Spin': PALETTE['highlight'], 'Pace/Other': PALETTE['blue']},
                    title="Death-over economy vs. wickets: spin bowlers trade runs for strikes differently than pace")
fig11.update_xaxes(title="Economy rate (overs 16-20)")
fig11.update_yaxes(title="Wickets taken (overs 16-20)")
fig11.show()


### Q12 — How does the average winning margin (by runs) differ by city?

*Group comparison across a spatial dimension.*


In [ ]:
runs_wins = matches[matches['win_by'] == 'runs']
city_margins = runs_wins.groupby('city')['win_margin'].mean().reset_index()
city_counts = runs_wins['city'].value_counts()
city_margins = city_margins[city_margins['city'].isin(city_counts[city_counts >= 10].index)]
city_margins = city_margins.sort_values('win_margin', ascending=False)

fig12 = px.bar(city_margins, x='win_margin', y='city', orientation='h', template=TEMPLATE,
               title="Average winning margin (by runs) differs notably by city")
fig12.update_traces(marker_color=PALETTE['blue'])
fig12.update_xaxes(title="Avg win margin (runs)")
fig12.update_yaxes(title="")
fig12.show()


## 5. Key Takeaways

*(Replace with your own conclusions once you've reviewed the figures above — this section feeds directly into the presentation deck.)*

- **Strategy has shifted**: chasing has become the statistically stronger approach over the seasons (Q1, Q3).
- **Toss matters, but unevenly**: its influence on match outcome depends heavily on venue and team (Q2, Q9).
- **Early aggression pays off**: strong powerplay batting is associated with winning (Q4).
- **Auction price ≠ performance**: market valuation is a weak signal for on-field output (Q5).
- **The talent gap is closing**: uncapped players have narrowed the strike-rate gap with capped internationals (Q6).
- **Context-dependent bowling**: death-over effectiveness and matchup strength vary meaningfully by bowler, opposition, and style (Q10, Q11).


## 6. Export cleaned tables for the Streamlit dashboard


In [ ]:
matches.to_csv('matches_clean.csv', index=False)
deliveries.to_csv('deliveries_clean.csv', index=False)

from google.colab import files as colab_files
colab_files.download('matches_clean.csv')
colab_files.download('deliveries_clean.csv')
